# Next-Hour Traffic Probe Count Forecasting

This notebook reproduces the preliminary modelling results used in the abstract working draft. Upload `selected_delhi_roads_hourly.csv` when prompted.


In [13]:
from google.colab import files
uploaded = files.upload()


Saving selected_delhi_roads_hourly.csv to selected_delhi_roads_hourly (4).csv


In [14]:
import io
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

print("Shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Roads:", df["street_name"].nunique())
df.head()


Shape: (4800, 12)
Date range: 2024-08-11 to 2024-08-30
Roads: 10


,date,street_name,hour,day_of_week,day_number,is_weekend,is_peak_hour,probe_count,segment_count,average_speed_limit,average_frc,total_distance
0,2024-08-11,Mahatma Gandhi Marg,0,Sunday,6,1,0,285435,1052,52.443,1.5722,62245.45
1,2024-08-11,Mahatma Gandhi Marg,1,Sunday,6,1,0,238136,1052,52.443,1.5722,62245.45
2,2024-08-11,Mahatma Gandhi Marg,2,Sunday,6,1,0,181220,1052,52.443,1.5722,62245.45
3,2024-08-11,Mahatma Gandhi Marg,3,Sunday,6,1,0,141749,1052,52.443,1.5722,62245.45
4,2024-08-11,Mahatma Gandhi Marg,4,Sunday,6,1,0,125713,1052,52.443,1.5722,62245.45


In [15]:
# Create the timestamp and arrange each road chronologically.
df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")
df = df.sort_values(["street_name", "datetime"]).reset_index(drop=True)

grouped = df.groupby("street_name", group_keys=False)

# Historical features available before forecasting the next hour.
df["lag_1"] = grouped["probe_count"].shift(1)
df["lag_24"] = grouped["probe_count"].shift(24)
df["rolling_mean_3"] = grouped["probe_count"].transform(lambda s: s.shift(1).rolling(3).mean())
df["rolling_mean_24"] = grouped["probe_count"].transform(lambda s: s.shift(1).rolling(24).mean())

# The target is the following hour's aggregated probe count for the same road.
df["target_next_hour"] = grouped["probe_count"].shift(-1)
df["target_datetime"] = df["datetime"] + pd.Timedelta(hours=1)
df["target_hour"] = df["target_datetime"].dt.hour
df["target_day_number"] = df["target_datetime"].dt.dayofweek
df["target_is_weekend"] = (df["target_day_number"] >= 5).astype(int)
df["target_is_peak_hour"] = df["target_hour"].isin([7, 8, 9, 17, 18, 19, 20]).astype(int)

model_df = df.dropna(subset=[
    "lag_1", "lag_24", "rolling_mean_3", "rolling_mean_24", "target_next_hour"
]).copy()

print("Model-ready records:", len(model_df))
print("First target time:", model_df["target_datetime"].min())
print("Last target time:", model_df["target_datetime"].max())


Model-ready records: 4550
First target time: 2024-08-12 01:00:00
Last target time: 2024-08-30 23:00:00


In [16]:
# Chronological holdout: the final four days are reserved for testing.
split_time = pd.Timestamp("2024-08-27 00:00:00")
train_df = model_df[model_df["target_datetime"] < split_time].copy()
test_df = model_df[model_df["target_datetime"] >= split_time].copy()

print("Training records:", len(train_df))
print("Testing records:", len(test_df))
print("Training target range:", train_df["target_datetime"].min(), "to", train_df["target_datetime"].max())
print("Testing target range:", test_df["target_datetime"].min(), "to", test_df["target_datetime"].max())


Training records: 3590
Testing records: 960
Training target range: 2024-08-12 01:00:00 to 2024-08-26 23:00:00
Testing target range: 2024-08-27 00:00:00 to 2024-08-30 23:00:00


In [17]:
categorical_features = ["street_name"]
numeric_features = [
    "probe_count", "lag_1", "lag_24", "rolling_mean_3", "rolling_mean_24",
    "target_hour", "target_day_number", "target_is_weekend", "target_is_peak_hour",
    "segment_count", "average_speed_limit", "average_frc", "total_distance"
]

feature_columns = categorical_features + numeric_features
X_train = train_df[feature_columns]
y_train = train_df["target_next_hour"]
X_test = test_df[feature_columns]
y_test = test_df["target_next_hour"]

preprocessor = ColumnTransformer([
    ("road", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ("numeric", "passthrough", numeric_features),
])

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.03, max_depth=3,
        min_samples_leaf=3, loss="huber", random_state=42
    ),
}

def evaluate(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R2": r2_score(actual, predicted),
    }

results = {}
fitted_models = {}

# Naive baseline: the next hour is predicted as equal to the current hour.
baseline_predictions = test_df["probe_count"].to_numpy()
results["Naive Current-Hour Baseline"] = evaluate(y_test, baseline_predictions)

for model_name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    results[model_name] = evaluate(y_test, predictions)
    fitted_models[model_name] = pipeline

results_df = pd.DataFrame(results).T
base_mae = results_df.loc["Naive Current-Hour Baseline", "MAE"]
base_rmse = results_df.loc["Naive Current-Hour Baseline", "RMSE"]
results_df["MAE Improvement vs Baseline (%)"] = (1 - results_df["MAE"] / base_mae) * 100
results_df["RMSE Improvement vs Baseline (%)"] = (1 - results_df["RMSE"] / base_rmse) * 100

results_df.round(4)


,MAE,RMSE,R2,MAE Improvement vs Baseline (%),RMSE Improvement vs Baseline (%)
Naive Current-Hour Baseline,14436.7229,25877.1157,0.9593,0.0000,0.0000
Random Forest,6915.0237,12052.5489,0.9912,52.1012,53.4239
Gradient Boosting,8093.8417,15162.5037,0.9860,43.9357,41.4057


In [18]:
best_model_name = results_df["MAE"].idxmin()
best_model = fitted_models[best_model_name]

print("Best model:", best_model_name)
print(results_df.loc[best_model_name].round(4))

results_df.to_csv("preliminary_model_results.csv")
joblib.dump(best_model, "best_next_hour_probe_model.joblib")

files.download("preliminary_model_results.csv")
files.download("best_next_hour_probe_model.joblib")


Best model: Random Forest
MAE                                  6915.0237
RMSE                                12052.5489
R2                                      0.9912
MAE Improvement vs Baseline (%)        52.1012
RMSE Improvement vs Baseline (%)       53.4239
Name: Random Forest, dtype: float64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
print("Uploaded files:", list(uploaded.keys()))

for name, content in uploaded.items():
    print("\nFilename:", name)
    print("Size:", len(content), "bytes")
    print("Beginning of file:")
    print(content[:300].decode("utf-8", errors="replace"))


Uploaded files: ['selected_delhi_roads_hourly (4).csv']

Filename: selected_delhi_roads_hourly (4).csv
Size: 376264 bytes
Beginning of file:
date,street_name,hour,day_of_week,day_number,is_weekend,is_peak_hour,probe_count,segment_count,average_speed_limit,average_frc,total_distance
2024-08-11,Mahatma Gandhi Marg,0,Sunday,6,1,0,285435,1052,52.443,1.5722,62245.45
2024-08-11,Mahatma Gandhi Marg,1,Sunday,6,1,0,238136,1052,52.443,1.5722,622


In [20]:
# Create the timestamp and arrange each road chronologically.
df["datetime"] = (
    pd.to_datetime(df["date"])
    + pd.to_timedelta(df["hour"], unit="h")
)

df = (
    df.sort_values(["street_name", "datetime"])
    .reset_index(drop=True)
)

grouped = df.groupby(
    "street_name",
    group_keys=False
)

# Historical features available before forecasting the next hour.
df["lag_1"] = grouped["probe_count"].shift(1)
df["lag_24"] = grouped["probe_count"].shift(24)

df["rolling_mean_3"] = grouped["probe_count"].transform(
    lambda series: series.shift(1).rolling(3).mean()
)

df["rolling_mean_24"] = grouped["probe_count"].transform(
    lambda series: series.shift(1).rolling(24).mean()
)

# Target: the following hour's aggregated probe count.
df["target_next_hour"] = grouped["probe_count"].shift(-1)

df["target_datetime"] = (
    df["datetime"] + pd.Timedelta(hours=1)
)

df["target_hour"] = df["target_datetime"].dt.hour

df["target_day_number"] = (
    df["target_datetime"].dt.dayofweek
)

df["target_is_weekend"] = (
    df["target_day_number"] >= 5
).astype(int)

df["target_is_peak_hour"] = (
    df["target_hour"].isin(
        [7, 8, 9, 17, 18, 19, 20]
    )
).astype(int)

model_df = df.dropna(
    subset=[
        "lag_1",
        "lag_24",
        "rolling_mean_3",
        "rolling_mean_24",
        "target_next_hour",
    ]
).copy()

print("Model-ready records:", len(model_df))
print(
    "First target time:",
    model_df["target_datetime"].min()
)
print(
    "Last target time:",
    model_df["target_datetime"].max()
)

Model-ready records: 4550
First target time: 2024-08-12 01:00:00
Last target time: 2024-08-30 23:00:00


In [21]:
# Reserve the final four days for testing.
# Earlier observations are used for training.

split_time = pd.Timestamp("2024-08-27 00:00:00")

train_df = model_df[
    model_df["target_datetime"] < split_time
].copy()

test_df = model_df[
    model_df["target_datetime"] >= split_time
].copy()

print("Training records:", len(train_df))
print("Testing records:", len(test_df))

print(
    "Training target range:",
    train_df["target_datetime"].min(),
    "to",
    train_df["target_datetime"].max()
)

print(
    "Testing target range:",
    test_df["target_datetime"].min(),
    "to",
    test_df["target_datetime"].max()
)

print(
    "Training roads:",
    train_df["street_name"].nunique()
)

print(
    "Testing roads:",
    test_df["street_name"].nunique()
)

Training records: 3590
Testing records: 960
Training target range: 2024-08-12 01:00:00 to 2024-08-26 23:00:00
Testing target range: 2024-08-27 00:00:00 to 2024-08-30 23:00:00
Training roads: 10
Testing roads: 10


In [22]:
categorical_features = [
    "street_name"
]

numeric_features = [
    "probe_count",
    "lag_1",
    "lag_24",
    "rolling_mean_3",
    "rolling_mean_24",
    "target_hour",
    "target_day_number",
    "target_is_weekend",
    "target_is_peak_hour",
    "segment_count",
    "average_speed_limit",
    "average_frc",
    "total_distance",
]

feature_columns = (
    categorical_features
    + numeric_features
)

X_train = train_df[feature_columns]
y_train = train_df["target_next_hour"]

X_test = test_df[feature_columns]
y_test = test_df["target_next_hour"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "road",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        ),
    ]
)

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        min_samples_leaf=3,
        loss="huber",
        random_state=42
    ),
}


def evaluate_model(actual, predicted):
    return {
        "MAE": mean_absolute_error(
            actual,
            predicted
        ),

        "RMSE": mean_squared_error(
            actual,
            predicted
        ) ** 0.5,

        "R2": r2_score(
            actual,
            predicted
        ),
    }


results = {}
fitted_models = {}

# Naive baseline:
# Predict that next-hour traffic will equal
# the current-hour probe count.
baseline_predictions = (
    test_df["probe_count"].to_numpy()
)

results[
    "Naive Current-Hour Baseline"
] = evaluate_model(
    y_test,
    baseline_predictions
)

for model_name, estimator in models.items():

    print("Training:", model_name)

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                estimator
            ),
        ]
    )

    pipeline.fit(
        X_train,
        y_train
    )

    predictions = pipeline.predict(
        X_test
    )

    results[model_name] = evaluate_model(
        y_test,
        predictions
    )

    fitted_models[model_name] = pipeline

    print(
        model_name,
        "completed."
    )

results_df = pd.DataFrame(
    results
).T

baseline_mae = results_df.loc[
    "Naive Current-Hour Baseline",
    "MAE"
]

baseline_rmse = results_df.loc[
    "Naive Current-Hour Baseline",
    "RMSE"
]

results_df[
    "MAE Improvement vs Baseline (%)"
] = (
    1
    - results_df["MAE"]
    / baseline_mae
) * 100

results_df[
    "RMSE Improvement vs Baseline (%)"
] = (
    1
    - results_df["RMSE"]
    / baseline_rmse
) * 100

print("\nMODEL COMPARISON")
print("=" * 90)

print(
    results_df
    .round(4)
    .to_string()
)

Training: Random Forest
Random Forest completed.
Training: Gradient Boosting
Gradient Boosting completed.

MODEL COMPARISON
                                    MAE        RMSE      R2  MAE Improvement vs Baseline (%)  RMSE Improvement vs Baseline (%)
Naive Current-Hour Baseline  14436.7229  25877.1157  0.9593                           0.0000                            0.0000
Random Forest                 6915.0237  12052.5489  0.9912                          52.1012                           53.4239
Gradient Boosting             8093.8417  15162.5037  0.9860                          43.9357                           41.4057


In [23]:
# Select the best-performing model.
best_model_name = "Random Forest"
best_model = fitted_models[best_model_name]

# Generate final test predictions.
best_predictions = best_model.predict(X_test)

# Save the trained pipeline.
model_filename = "random_forest_next_hour_traffic_model.joblib"
joblib.dump(best_model, model_filename)

# Save the model-comparison table.
results_filename = "model_comparison_results.csv"
results_df.round(4).to_csv(
    results_filename,
    index_label="Model"
)

# Save actual and predicted testing values.
predictions_df = test_df[
    [
        "target_datetime",
        "street_name",
        "target_next_hour"
    ]
].copy()

predictions_df["predicted_next_hour"] = best_predictions
predictions_df["absolute_error"] = np.abs(
    predictions_df["target_next_hour"]
    - predictions_df["predicted_next_hour"]
)

predictions_filename = "random_forest_test_predictions.csv"
predictions_df.to_csv(
    predictions_filename,
    index=False
)

print("Best model:", best_model_name)
print("Saved model:", model_filename)
print("Saved comparison:", results_filename)
print("Saved predictions:", predictions_filename)

files.download(model_filename)
files.download(results_filename)
files.download(predictions_filename)

Best model: Random Forest
Saved model: random_forest_next_hour_traffic_model.joblib
Saved comparison: model_comparison_results.csv
Saved predictions: random_forest_test_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>